In [1]:
import pickle
import os
import numpy as np
print("PYTHONPATH:", os.environ.get('PYTHONPATH'))
print("PATH:", os.environ.get('PATH'))
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn import preprocessing
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandas as pd
import phate
import math
import random
import gc
import scprep
from datetime import datetime, time
from matplotlib.animation import ImageMagickWriter
import matplotlib.animation as animation
import zipfile
from urllib.request import urlopen
import scipy.stats as st
from scipy.stats import norm
from scipy.stats import gaussian_kde
from scipy.stats import kde
from scipy.stats import binned_statistic
from scipy.stats import f_oneway
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['pdf.fonttype'] = 42
print(sns.__version__)
from anndata import AnnData
import scanpy as sc
from delve import *
import anndata as ad
from sklearn.preprocessing import MinMaxScaler
from kh import sketch
from sklearn.cluster import KMeans
import umap
print(sc.__version__)
today = datetime.now().strftime("%m%d%Y-%H%M")

PYTHONPATH: None
PATH: c:\Users\Purvis_Jane\.conda\envs\python_3_7;C:\Users\Purvis_Jane\.conda\envs\python_3_7;C:\Users\Purvis_Jane\.conda\envs\python_3_7\Library\mingw-w64\bin;C:\Users\Purvis_Jane\.conda\envs\python_3_7\Library\usr\bin;C:\Users\Purvis_Jane\.conda\envs\python_3_7\Library\bin;C:\Users\Purvis_Jane\.conda\envs\python_3_7\Scripts;C:\Users\Purvis_Jane\.conda\envs\python_3_7\bin;C:\ProgramData\anaconda3\condabin;C:\ProgramData\anaconda3;C:\ProgramData\anaconda3\Library\mingw-w64\bin;C:\ProgramData\anaconda3\Library\usr\bin;C:\ProgramData\anaconda3\Library\bin;C:\ProgramData\anaconda3\Scripts;C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v11.2\bin;C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v11.2\libnvvp;.;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0;C:\WINDOWS\System32\OpenSSH;C:\Program Files\NVIDIA Corporation\Nsight Compute 2020.3.0;C:\Program Files (x86)\NVIDIA Corporation\PhysX\Common;C:\Users\Purvis_Jane

In [2]:
### full_dir should be the directory that contains the output of your "calculate cell properties" notebook. Called cell_data by default
### well_list should be every well in the dataset that you intend to comDine into a single adata object for analysis

full_dir = r'D:\Sarah\experiment_4_2\cell_data'
#well_list = ['D02','D03','D04','D05','D06','D07','D08','D09','D10','D11','E02','E03','E04','E05','E06','E07','E08','E09','E10','E11'] #LPS863
#well_list = ['B02','B03','B04','B05','B06','B07','B08','B09','B10','B11','C02','C03','C04','C05','C06','C07','C08','C09','C10','C11'] #LPS246
#well_list = ['B02','B09','B10','B11','C02','C09','C10','C11'] 
#Primary Tumor 8577FL July 2025 
#well_list = ['B02','B03','B04','B05','B06','B07','B08','B09','B10','B11','C02','C03','C04','C05','C06','C07','C08','C09','C10','C11','D02','D03','D04','D05','D06','D07','D08','D09','D10','D11','E02','E03','E04','E05','E06','E07','E08','E09','E10','E11','F02','F03','F04','F05','F06','F07','F08','F09','F10','F11']
#Primary Tumor 8675CL August 2025
#well_list = ['B02','B03','B04','B05','B06','B07','B08','B09','B10','B11','C02','C03','C04','C05','C06','C07','C08','C09','C10','C11','D02','D03','D04','D05','D06','D07','D08','D09','D10','D11','F10','F11','G02','G03','G04','G05','G06','G07','G08','G09','G10','G11']
#well_list = ['B02','B03','B04','B05','B06','B07','C02','C03','C04','C05','C06','C07','C08','D02','D04','D05','D06','D07','D08','E02','E03','E04','E05','E06','E07','E08','F03','F04','F05','F06','F07','F08','G02','G03','G04','G05','G06','G07','G08',]
well_list = ['B02','B03','B05','B07','B09','C02','C03','C05','C07','C09','D02','D03','D05','D07','D09','E02','E03','E05','E07','E09','F02','F03','F05','F07','F09','G02','G03','G05','G07','G09']


In [3]:
# Definition to Normalize the dataframe by z-score

def standardizeColumns(df):
    df = df.copy()
    df.iloc[:,:] = df.iloc[:,:].apply(lambda x: (x-x.mean())/ x.std(), axis=0)
    return df

Clara

In [ ]:
#Builds a dataframe for each well defined in well_list, then outputs the length of that list. The number output is how many cells
#were found in the well

fullest_df = pd.DataFrame()
for well in well_list:
    print(f'starting Well {well}')
    full_df = pd.read_csv(os.path.join(full_dir,f'cell_data_{well}_df.csv'), sep=',') 

   ### This part here is optional, you can add important metadata information at this step. Treatment and Sample ID are shown as examples
    if 'B' in well and ('02' in well):
        full_df['dose'] = '0uM'
        full_df['treatment'] = 'NT' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('03' in well):
        full_df['dose'] = '100nM'
        full_df['treatment'] = 'Fulv'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'B' in well and ('05' in well):
        full_df['dose'] = '100nM + 100nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'B' in well and ('07' in well):
        full_df['dose'] = '100nM + 200nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 4
        #full_df['Group'] = 1
    if 'B' in well and ('09' in well):
        full_df['dose'] = '100nM + 500nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1

    if 'C' in well and ('02' in well):
        full_df['dose'] = '0uM'
        full_df['treatment'] = 'NT' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'C' in well and ('03' in well):
        full_df['dose'] = '100nM'
        full_df['treatment'] = 'Fulv'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'C' in well and ('05' in well):
        full_df['dose'] = '100nM + 100nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'C' in well and ('07' in well):
        full_df['dose'] = '100nM + 200nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 4
        #full_df['Group'] = 1
    if 'C' in well and ('09' in well):
        full_df['dose'] = '100nM + 500nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1
    
    if 'D' in well and ('02' in well):
        full_df['dose'] = '0uM'
        full_df['treatment'] = 'NT' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'D' in well and ('03' in well):
        full_df['dose'] = '100nM'
        full_df['treatment'] = 'Fulv'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'D' in well and ('05' in well):
        full_df['dose'] = '100nM + 100nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'D' in well and ('07' in well):
        full_df['dose'] = '100nM + 200nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 4
        #full_df['Group'] = 1
    if 'D' in well and ('09' in well):
        full_df['dose'] = '100nM + 500nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1
    
    if 'E' in well and ('02' in well):
        full_df['dose'] = '10nM'
        full_df['treatment'] = 'Gemc' 
        full_df['sample_ID'] = 6
       # full_df['Group'] = 1
    if 'E' in well and ('03' in well):
        full_df['dose'] = '10nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv'
        full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    if 'E' in well and ('05' in well):
        full_df['dose'] = '10nM + 100nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'E' in well and ('07' in well):
        full_df['dose'] = '10nM + 100nM + 200nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 9
        #full_df['Group'] = 1
    if 'E' in well and ('09' in well):
        full_df['dose'] = '10nM + 100nM + 500nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 10
        #full_df['Group'] = 1

    if 'F' in well and ('02' in well):
        full_df['dose'] = '10nM'
        full_df['treatment'] = 'Gemc' 
        full_df['sample_ID'] = 6
       # full_df['Group'] = 1
    if 'F' in well and ('03' in well):
        full_df['dose'] = '10nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv'
        full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    if 'F' in well and ('05' in well):
        full_df['dose'] = '10nM + 100nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'F' in well and ('07' in well):
        full_df['dose'] = '10nM + 100nM + 200nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 9
        #full_df['Group'] = 1
    if 'F' in well and ('09' in well):
        full_df['dose'] = '10nM + 100nM + 500nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 10
        #full_df['Group'] = 1
    
    if 'G' in well and ('02' in well):
        full_df['dose'] = '10nM'
        full_df['treatment'] = 'Gemc' 
        full_df['sample_ID'] = 6
       # full_df['Group'] = 1
    if 'G' in well and ('03' in well):
        full_df['dose'] = '10nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv'
        full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    if 'G' in well and ('05' in well):
        full_df['dose'] = '10nM + 100nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'G' in well and ('07' in well):
        full_df['dose'] = '10nM + 100nM + 200nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 9
        #full_df['Group'] = 1
    if 'G' in well and ('09' in well):
        full_df['dose'] = '10nM + 100nM + 500nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 10
        #full_df['Group'] = 1

    exec(f'well{well}_df = full_df.copy()')
    fullest_df = pd.concat([fullest_df, full_df], ignore_index = True)
    print(len(full_df))
    print(len(fullest_df))

### You may find that you need to force some of the columns added this way into specific data types. You can do that here.
# Convert sample_ID column to categorical data type
fullest_df['sample_ID'] = fullest_df['sample_ID'].astype('category')
fullest_df['dose'] = fullest_df['dose'].astype(str)
fullest_df['treatment'] = fullest_df['treatment'].astype(str)

Sarah's Exp 4_2

In [ ]:
#Builds a dataframe for each well defined in well_list, then outputs the length of that list. The number output is how many cells
#were found in the well

fullest_df = pd.DataFrame()
for well in well_list:
    print(f'starting Well {well}')
    full_df = pd.read_csv(os.path.join(full_dir,f'cell_data_{well}_df.csv'), sep=',') 

   ### This part here is optional, you can add important metadata information at this step. Treatment and Sample ID are shown as examples
    if 'B' in well and ('02' in well):
        full_df['dose'] = '0uM'
        full_df['treatment'] = 'NT' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('03' in well):
        full_df['dose'] = '100nM'
        full_df['treatment'] = 'Fulv'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'B' in well and ('05' in well):
        full_df['dose'] = '100nM + 100nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'B' in well and ('07' in well):
        full_df['dose'] = '100nM + 200nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 4
        #full_df['Group'] = 1
    if 'B' in well and ('09' in well):
        full_df['dose'] = '100nM + 500nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1

    if 'C' in well and ('02' in well):
        full_df['dose'] = '0uM'
        full_df['treatment'] = 'NT' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'C' in well and ('03' in well):
        full_df['dose'] = '100nM'
        full_df['treatment'] = 'Fulv'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'C' in well and ('05' in well):
        full_df['dose'] = '100nM + 100nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'C' in well and ('07' in well):
        full_df['dose'] = '100nM + 200nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 4
        #full_df['Group'] = 1
    if 'C' in well and ('09' in well):
        full_df['dose'] = '100nM + 500nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1
    
    if 'D' in well and ('02' in well):
        full_df['dose'] = '0uM'
        full_df['treatment'] = 'NT' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'D' in well and ('03' in well):
        full_df['dose'] = '100nM'
        full_df['treatment'] = 'Fulv'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'D' in well and ('05' in well):
        full_df['dose'] = '100nM + 100nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'D' in well and ('07' in well):
        full_df['dose'] = '100nM + 200nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 4
        #full_df['Group'] = 1
    if 'D' in well and ('09' in well):
        full_df['dose'] = '100nM + 500nM'
        full_df['treatment'] = 'Fulv + Palbo'
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1
    
    if 'E' in well and ('02' in well):
        full_df['dose'] = '10nM'
        full_df['treatment'] = 'Gemc' 
        full_df['sample_ID'] = 6
       # full_df['Group'] = 1
    if 'E' in well and ('03' in well):
        full_df['dose'] = '10nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv'
        full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    if 'E' in well and ('05' in well):
        full_df['dose'] = '10nM + 100nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'E' in well and ('07' in well):
        full_df['dose'] = '10nM + 100nM + 200nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 9
        #full_df['Group'] = 1
    if 'E' in well and ('09' in well):
        full_df['dose'] = '10nM + 100nM + 500nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 10
        #full_df['Group'] = 1

    if 'F' in well and ('02' in well):
        full_df['dose'] = '10nM'
        full_df['treatment'] = 'Gemc' 
        full_df['sample_ID'] = 6
       # full_df['Group'] = 1
    if 'F' in well and ('03' in well):
        full_df['dose'] = '10nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv'
        full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    if 'F' in well and ('05' in well):
        full_df['dose'] = '10nM + 100nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'F' in well and ('07' in well):
        full_df['dose'] = '10nM + 100nM + 200nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 9
        #full_df['Group'] = 1
    if 'F' in well and ('09' in well):
        full_df['dose'] = '10nM + 100nM + 500nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 10
        #full_df['Group'] = 1
    
    if 'G' in well and ('02' in well):
        full_df['dose'] = '10nM'
        full_df['treatment'] = 'Gemc' 
        full_df['sample_ID'] = 6
       # full_df['Group'] = 1
    if 'G' in well and ('03' in well):
        full_df['dose'] = '10nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv'
        full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    if 'G' in well and ('05' in well):
        full_df['dose'] = '10nM + 100nM + 100nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'G' in well and ('07' in well):
        full_df['dose'] = '10nM + 100nM + 200nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 9
        #full_df['Group'] = 1
    if 'G' in well and ('09' in well):
        full_df['dose'] = '10nM + 100nM + 500nM'
        full_df['treatment'] = 'Gemc + Fulv + Palbo'
        full_df['sample_ID'] = 10
        #full_df['Group'] = 1

    exec(f'well{well}_df = full_df.copy()')
    fullest_df = pd.concat([fullest_df, full_df], ignore_index = True)
    print(len(full_df))
    print(len(fullest_df))

### You may find that you need to force some of the columns added this way into specific data types. You can do that here.
# Convert sample_ID column to categorical data type
fullest_df['sample_ID'] = fullest_df['sample_ID'].astype('category')
fullest_df['dose'] = fullest_df['dose'].astype(str)
fullest_df['treatment'] = fullest_df['treatment'].astype(str)

starting Well B02
8629
8629
starting Well B03
21526
30155
starting Well B05
15447
45602
starting Well B07
15300
60902
starting Well B09
13223
74125
starting Well C02
9718
83843
starting Well C03
14694
98537
starting Well C05
15762
114299
starting Well C07
13752
128051
starting Well C09
7787
135838
starting Well D02
7303
143141
starting Well D03
16223
159364
starting Well D05
9598
168962
starting Well D07
15031
183993
starting Well D09
13995
197988
starting Well E02
3703
201691
starting Well E03
15264
216955
starting Well E05
13588
230543
starting Well E07
12202
242745
starting Well E09
7313
250058
starting Well F02
5241
255299
starting Well F03
13319
268618
starting Well F05
17952
286570
starting Well F07
13088
299658
starting Well F09
9394
309052
starting Well G02
4836
313888
starting Well G03
17448
331336
starting Well G05
15114
346450
starting Well G07
9944
356394
starting Well G09
7278
363672


CVT + Tagto + Alp LPS 246

In [ ]:
#Builds a dataframe for each well defined in well_list, then outputs the length of that list. The number output is how many cells
#were found in the well

fullest_df = pd.DataFrame()
for well in well_list:
    print(f'starting Well {well}')
    full_df = pd.read_csv(os.path.join(full_dir,f'cell_data_{well}_df.csv'), sep=',') 

   ### This part here is optional, you can add important metadata information at this step. Treatment and Sample ID are shown as examples
    if 'B' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('03' in well):
        full_df['dose'] = '0.01 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'B' in well and ('04' in well):
        full_df['dose'] = '0.05 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'B' in well and ('05' in well):
        full_df['dose'] = '0.1 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 4
       # full_df['Group'] = 1
    if 'B' in well and ('06' in well):
        full_df['dose'] = '0.5 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1
    if 'B' in well and ('07' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 6
        #full_df['Group'] = 1
    if 'B' in well and ('08' in well):
        full_df['dose'] = '2 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 7
        #full_df['Group'] = 1
    if 'B' in well and ('09' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'B' in well and ('10' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 9
       # full_df['Group'] = 1  
    if 'B' in well and ('11' in well):
        full_df['dose'] = '20 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1
    
    if 'C' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'C' in well and ('03' in well):
        full_df['dose'] = '0.01 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'C' in well and ('04' in well):
        full_df['dose'] = '0.05 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'C' in well and ('05' in well):
        full_df['dose'] = '0.1 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 4
       # full_df['Group'] = 1
    if 'C' in well and ('06' in well):
        full_df['dose'] = '0.5 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1
    if 'C' in well and ('07' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 6
        #full_df['Group'] = 1
    if 'C' in well and ('08' in well):
        full_df['dose'] = '2 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 7
        #full_df['Group'] = 1
    if 'C' in well and ('09' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'C' in well and ('10' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 9
       # full_df['Group'] = 1  
    if 'C' in well and ('11' in well):
        full_df['dose'] = '20 uM'
        full_df['treatment'] = 'CVT'
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1

    if 'D' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'D' in well and ('03' in well):
        full_df['dose'] = '0.01 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 11
       # full_df['Group'] = 1
    if 'D' in well and ('04' in well):
        full_df['dose'] = '0.05 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 12
       # full_df['Group'] = 1
    if 'D' in well and ('05' in well):
        full_df['dose'] = '0.1 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 13
        # full_df['Group'] = 1
    if 'D' in well and ('06' in well):
        full_df['dose'] = '0.25 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 14
        # full_df['Group'] = 1
    if 'D' in well and ('07' in well):
        full_df['dose'] = '0.5 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 15
        #full_df['Group'] = 2
    if 'D' in well and ('08' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 16
        #full_df['Group'] = 2
    if 'D' in well and ('09' in well):
        full_df['dose'] = '2 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 17
       # full_df['Group'] = 2   
    if 'D' in well and ('10' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 18
       # full_df['Group'] = 1  
    if 'D' in well and ('11' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 19
       # full_df['Group'] = 1 

    if 'E' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'E' in well and ('03' in well):
        full_df['dose'] = '0.01 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 11
       # full_df['Group'] = 1
    if 'E' in well and ('04' in well):
        full_df['dose'] = '0.05 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 12
       # full_df['Group'] = 1
    if 'E' in well and ('05' in well):
        full_df['dose'] = '0.1 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 13
        # full_df['Group'] = 1
    if 'E' in well and ('06' in well):
        full_df['dose'] = '0.25 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 14
        # full_df['Group'] = 1
    if 'E' in well and ('07' in well):
        full_df['dose'] = '0.5 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 15
        #full_df['Group'] = 2
    if 'E' in well and ('08' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 16
        #full_df['Group'] = 2
    if 'E' in well and ('09' in well):
        full_df['dose'] = '2 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 17
       # full_df['Group'] = 2   
    if 'E' in well and ('10' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 18
       # full_df['Group'] = 1  
    if 'E' in well and ('11' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Tagto'
        full_df['sample_ID'] = 19
       # full_df['Group'] = 1 

    if 'F' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'F' in well and ('03' in well):
        full_df['dose'] = '0.01 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 20
       # full_df['Group'] = 1
    if 'F' in well and ('04' in well):
        full_df['dose'] = '0.03 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 21
       # full_df['Group'] = 1
    if 'F' in well and ('05' in well):
        full_df['dose'] = '0.1 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 22
       # full_df['Group'] = 1
    if 'F' in well and ('06' in well):
        full_df['dose'] = '0.3 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 23
        #full_df['Group'] = 1
    if 'F' in well and ('07' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 24
        #full_df['Group'] = 1
    if 'F' in well and ('08' in well):
        full_df['dose'] = '3 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 25
        #full_df['Group'] = 1
    if 'F' in well and ('09' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 26
       # full_df['Group'] = 1
    if 'F' in well and ('10' in well):
        full_df['dose'] = '20 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 27
       # full_df['Group'] = 1  
    if 'F' in well and ('11' in well):
        full_df['dose'] = '30 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 28
       # full_df['Group'] = 1

    if 'G' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'G' in well and ('03' in well):
        full_df['dose'] = '0.01 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 20
       # full_df['Group'] = 1
    if 'G' in well and ('04' in well):
        full_df['dose'] = '0.03 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 21
       # full_df['Group'] = 1
    if 'G' in well and ('05' in well):
        full_df['dose'] = '0.1 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 22
       # full_df['Group'] = 1
    if 'G' in well and ('06' in well):
        full_df['dose'] = '0.3 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 23
        #full_df['Group'] = 1
    if 'G' in well and ('07' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 24
        #full_df['Group'] = 1
    if 'G' in well and ('08' in well):
        full_df['dose'] = '3 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 25
        #full_df['Group'] = 1
    if 'G' in well and ('09' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 26
       # full_df['Group'] = 1
    if 'G' in well and ('10' in well):
        full_df['dose'] = '20 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 27
       # full_df['Group'] = 1  
    if 'G' in well and ('11' in well):
        full_df['dose'] = '30 uM'
        full_df['treatment'] = 'Alp'
        full_df['sample_ID'] = 28
       # full_df['Group'] = 1
       
    exec(f'well{well}_df = full_df.copy()')
    fullest_df = pd.concat([fullest_df, full_df], ignore_index = True)
    print(len(full_df))
    print(len(fullest_df))

### You may find that you need to force some of the columns added this way into specific data types. You can do that here.
# Convert sample_ID column to categorical data type
fullest_df['sample_ID'] = fullest_df['sample_ID'].astype('category')
fullest_df['dose'] = fullest_df['dose'].astype(str)
fullest_df['treatment'] = fullest_df['treatment'].astype(str)

starting Well B02
7343
7343
starting Well B03
7419
14762
starting Well B04
7930
22692
starting Well B05
8040
30732
starting Well B06
7991
38723
starting Well B07
8134
46857
starting Well B08
8102
54959
starting Well B09
7524
62483
starting Well B10
4856
67339
starting Well B11
4329
71668
starting Well C02
7634
79302
starting Well C03
7884
87186
starting Well C04
7960
95146
starting Well C05
8238
103384
starting Well C06
7880
111264
starting Well C07
8222
119486
starting Well C08
8365
127851
starting Well C09
7795
135646
starting Well C10
5072
140718
starting Well C11
4523
145241
starting Well D02
7037
152278
starting Well D03
7759
160037
starting Well D04
8054
168091
starting Well D05
8301
176392
starting Well D06
7830
184222
starting Well D07
6637
190859
starting Well D08
5256
196115
starting Well D09
4401
200516
starting Well D10
4805
205321
starting Well D11
4524
209845
starting Well E02
7638
217483
starting Well E03
7753
225236
starting Well E04
7965
233201
starting Well E05
8041
2

Primary Tumor 8577FL July 2025

In [4]:
#Builds a dataframe for each well defined in well_list, then outputs the length of that list. The number output is how many cells
#were found in the well

fullest_df = pd.DataFrame()
for well in well_list:
    print(f'starting Well {well}')
    full_df = pd.read_csv(os.path.join(full_dir,f'cell_data_{well}_df.csv'), sep=',') 

   ### This part here is optional, you can add important metadata information at this step. Treatment and Sample ID are shown as examples
    if 'B' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('03' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('04' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'B' in well and ('05' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'B' in well and ('06' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 3
        #full_df['Group'] = 1
    if 'B' in well and ('07' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 3
        #full_df['Group'] = 1
    if 'B' in well and ('08' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 4
        #full_df['Group'] = 1
    if 'B' in well and ('09' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 4
       # full_df['Group'] = 1
    if 'B' in well and ('10' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 5
       # full_df['Group'] = 1  
    if 'B' in well and ('11' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 5
       # full_df['Group'] = 1
    
    if 'C' in well and ('02' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 6
       # full_df['Group'] = 2
    if 'C' in well and ('03' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 6
    if 'C' in well and ('04' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 7
       # full_df['Group'] = 2
    if 'C' in well and ('05' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 7
       # full_df['Group'] = 2
    if 'C' in well and ('06' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 8
    if 'C' in well and ('07' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 8
        #full_df['Group'] = 2
    if 'C' in well and ('08' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 9
        #full_df['Group'] = 2
    if 'C' in well and ('09' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 9
       # full_df['Group'] = 2   
    if 'C' in well and ('10' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1  
    if 'C' in well and ('11' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1 

    if 'D' in well and ('02' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 11
       # full_df['Group'] = 2
    if 'D' in well and ('03' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 11
    if 'D' in well and ('04' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 12
       # full_df['Group'] = 2
    if 'D' in well and ('05' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 12
       # full_df['Group'] = 2
    if 'D' in well and ('06' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 13
    if 'D' in well and ('07' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 13
        #full_df['Group'] = 2
    if 'D' in well and ('08' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 14
        #full_df['Group'] = 2
    if 'D' in well and ('09' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 14
       # full_df['Group'] = 2   
    if 'D' in well and ('10' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 15
       # full_df['Group'] = 1  
    if 'D' in well and ('11' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 15
       # full_df['Group'] = 1 

    if 'E' in well and ('02' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 16
       # full_df['Group'] = 2
    if 'E' in well and ('03' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 16
    if 'E' in well and ('04' in well):
        full_df['dose'] = '10 nM + 1 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 17
       # full_df['Group'] = 2
    if 'E' in well and ('05' in well):
        full_df['dose'] = '10 nM + 5 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 18
       # full_df['Group'] = 2
    if 'E' in well and ('06' in well):
        full_df['dose'] = '10 nM + 10 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 19
    if 'E' in well and ('07' in well):
        full_df['dose'] = '50 nM + 1 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 20
        #full_df['Group'] = 2
    if 'E' in well and ('08' in well):
        full_df['dose'] = '50 nM + 5 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 21
        #full_df['Group'] = 2
    if 'E' in well and ('09' in well):
        full_df['dose'] = '50 nM  + 10 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 22
       # full_df['Group'] = 2   
    if 'E' in well and ('10' in well):
        full_df['dose'] = '250 nM + 1 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 23
       # full_df['Group'] = 1  
    if 'E' in well and ('11' in well):
        full_df['dose'] = '250 nM + 5 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 24
       # full_df['Group'] = 1    

    if 'F' in well and ('02' in well):
        full_df['dose'] = '250 nM + 10 uM'
        full_df['treatment'] = 'Palbo + Nutlin' 
        full_df['sample_ID'] = 25
       # full_df['Group'] = 2
    if 'F' in well and ('03' in well):
        full_df['dose'] = '10 nM + 1 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 26
    if 'F' in well and ('04' in well):
        full_df['dose'] = '10 nM + 5 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 27
       # full_df['Group'] = 2
    if 'F' in well and ('05' in well):
        full_df['dose'] = '10 nM + 10 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 28
       # full_df['Group'] = 2
    if 'F' in well and ('06' in well):
        full_df['dose'] = '50 nM + 1 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 29
    if 'F' in well and ('07' in well):
        full_df['dose'] = '50 nM + 5 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 30
        #full_df['Group'] = 2
    if 'F' in well and ('08' in well):
        full_df['dose'] = '50 nM + 10 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 31
        #full_df['Group'] = 2
    if 'F' in well and ('09' in well):
        full_df['dose'] = '250 nM + 1 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 32
       # full_df['Group'] = 2   
    if 'F' in well and ('10' in well):
        full_df['dose'] = '250 nM + 5 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 33
       # full_df['Group'] = 1  
    if 'F' in well and ('11' in well):
        full_df['dose'] = '250 nM + 10 uM'
        full_df['treatment'] = 'Abema + Nutlin' 
        full_df['sample_ID'] = 34
       # full_df['Group'] = 1    
       
    exec(f'well{well}_df = full_df.copy()')
    fullest_df = pd.concat([fullest_df, full_df], ignore_index = True)
    print(len(full_df))
    print(len(fullest_df))

### You may find that you need to force some of the columns added this way into specific data types. You can do that here.
# Convert sample_ID column to categorical data type
fullest_df['sample_ID'] = fullest_df['sample_ID'].astype('category')
fullest_df['dose'] = fullest_df['dose'].astype(str)
fullest_df['treatment'] = fullest_df['treatment'].astype(str)

starting Well B02
12698
12698
starting Well B03
12663
25361
starting Well B04
12923
38284
starting Well B05
11580
49864
starting Well B06
13063
62927
starting Well B07
13951
76878
starting Well B08
12992
89870
starting Well B09
11729
101599
starting Well B10
12447
114046
starting Well B11
13341
127387
starting Well C02
12932
140319
starting Well C03
13140
153459
starting Well C04
12714
166173
starting Well C05
13408
179581
starting Well C06
13403
192984
starting Well C07
13415
206399
starting Well C08
12972
219371
starting Well C09
13719
233090
starting Well C10
12890
245980
starting Well C11
13273
259253
starting Well D02
13903
273156
starting Well D03
12757
285913
starting Well D04
13961
299874
starting Well D05
13011
312885
starting Well D06
13282
326167
starting Well D07
13422
339589
starting Well D08
13305
352894
starting Well D09
12928
365822
starting Well D10
13068
378890
starting Well D11
13124
392014
starting Well E02
13472
405486
starting Well E03
11882
417368
starting Well E

Primary Tumor 8675CL August 2025

In [35]:
#Builds a dataframe for each well defined in well_list, then outputs the length of that list. The number output is how many cells
#were found in the well

fullest_df = pd.DataFrame()
for well in well_list:
    print(f'starting Well {well}')
    full_df = pd.read_csv(os.path.join(full_dir,f'cell_data_{well}_df.csv'), sep=',') 

   ### This part here is optional, you can add important metadata information at this step. Treatment and Sample ID are shown as examples
    if 'B' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('03' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('04' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'B' in well and ('05' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'B' in well and ('06' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 3
        #full_df['Group'] = 1
    if 'B' in well and ('07' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 3
        #full_df['Group'] = 1
    if 'B' in well and ('08' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 4
        #full_df['Group'] = 1
    if 'B' in well and ('09' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 4
       # full_df['Group'] = 1
    if 'B' in well and ('10' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 5
       # full_df['Group'] = 1  
    if 'B' in well and ('11' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 5
       # full_df['Group'] = 1
    
    if 'C' in well and ('02' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 6
       # full_df['Group'] = 2
    if 'C' in well and ('03' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 6
    if 'C' in well and ('04' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 7
       # full_df['Group'] = 2
    if 'C' in well and ('05' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Palbo' 
        full_df['sample_ID'] = 7
       # full_df['Group'] = 2
    if 'C' in well and ('06' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 8
    if 'C' in well and ('07' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 9
        #full_df['Group'] = 2
    if 'C' in well and ('08' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 9
        #full_df['Group'] = 2
    if 'C' in well and ('09' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 10
       # full_df['Group'] = 2   
    if 'C' in well and ('10' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1  
    if 'C' in well and ('11' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 11
       # full_df['Group'] = 1 

    if 'D' in well and ('02' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 11
       # full_df['Group'] = 2
    if 'D' in well and ('03' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 12
    if 'D' in well and ('04' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 12
       # full_df['Group'] = 2
    if 'D' in well and ('05' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 13
       # full_df['Group'] = 2
    if 'D' in well and ('06' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 14
    if 'D' in well and ('07' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 14
        #full_df['Group'] = 2
    if 'D' in well and ('08' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 15
        #full_df['Group'] = 2
    if 'D' in well and ('09' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 15
       # full_df['Group'] = 2   
    if 'D' in well and ('10' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 16
       # full_df['Group'] = 1  
    if 'D' in well and ('11' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 16
       # full_df['Group'] = 1  
       
    if 'F' in well and ('10' in well):
        full_df['dose'] = '1 um'
        full_df['treatment'] = 'CVT' 
        full_df['sample_ID'] = 17
       # full_df['Group'] = 1
    if 'F' in well and ('11' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'CVT' 
        full_df['sample_ID'] = 18
       # full_df['Group'] = 1
    
    if 'G' in well and ('02' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'CVT' 
        full_df['sample_ID'] = 19
       # full_df['Group'] = 2
    if 'G' in well and ('03' in well):
        full_df['dose'] = '20 uM'
        full_df['treatment'] = 'CVT' 
        full_df['sample_ID'] = 20
    if 'G' in well and ('04' in well):
        full_df['dose'] = '1 uM + 10 nM'
        full_df['treatment'] = 'CVT + Palbo' 
        full_df['sample_ID'] = 21
       # full_df['Group'] = 2
    if 'G' in well and ('05' in well):
        full_df['dose'] = '5 uM + 10 nM'
        full_df['treatment'] = 'CVT + Palbo' 
        full_df['sample_ID'] = 22
       # full_df['Group'] = 2
    if 'G' in well and ('06' in well):
        full_df['dose'] = '10 uM + 10 nM'
        full_df['treatment'] = 'CVT + Palbo' 
        full_df['sample_ID'] = 23
    if 'G' in well and ('07' in well):
        full_df['dose'] = '20 uM + 10 nM'
        full_df['treatment'] = 'CVT + Palbo' 
        full_df['sample_ID'] = 24
        #full_df['Group'] = 2
    if 'G' in well and ('08' in well):
        full_df['dose'] = '1 uM + 50 nM'
        full_df['treatment'] = 'CVT + Palbo' 
        full_df['sample_ID'] = 25
        #full_df['Group'] = 2
    if 'G' in well and ('09' in well):
        full_df['dose'] = '5 uM + 50 nM'
        full_df['treatment'] = 'CVT + Palbo' 
        full_df['sample_ID'] = 26
       # full_df['Group'] = 2   
    if 'G' in well and ('10' in well):
        full_df['dose'] = '10 uM + 50 nM'
        full_df['treatment'] = 'CVT + Palbo' 
        full_df['sample_ID'] = 27
       # full_df['Group'] = 1  
    if 'G' in well and ('11' in well):
        full_df['dose'] = '10 uM + 50 nM'
        full_df['treatment'] = 'CVT + Palbo' 
        full_df['sample_ID'] = 28
       # full_df['Group'] = 1 
       
    exec(f'well{well}_df = full_df.copy()')
    fullest_df = pd.concat([fullest_df, full_df], ignore_index = True)
    print(len(full_df))
    print(len(fullest_df))

### You may find that you need to force some of the columns added this way into specific data types. You can do that here.
# Convert sample_ID column to categorical data type
fullest_df['sample_ID'] = fullest_df['sample_ID'].astype('category')
fullest_df['dose'] = fullest_df['dose'].astype(str)
fullest_df['treatment'] = fullest_df['treatment'].astype(str)

starting Well B02
23964
23964
starting Well B03
23176
47140
starting Well B04
23825
70965
starting Well B05
22752
93717
starting Well B06
24164
117881
starting Well B07
22346
140227
starting Well B08
23330
163557
starting Well B09
23449
187006
starting Well B10
22071
209077
starting Well B11
23260
232337
starting Well C02
23859
256196
starting Well C03
23631
279827
starting Well C04
24827
304654
starting Well C05
23764
328418
starting Well C06
25222
353640
starting Well C07
24769
378409
starting Well C08
25230
403639
starting Well C09
25937
429576
starting Well C10
23747
453323
starting Well C11
23957
477280
starting Well D02
24854
502134
starting Well D03
23831
525965
starting Well D04
24223
550188
starting Well D05
24734
574922
starting Well D06
24337
599259
starting Well D07
25494
624753
starting Well D08
23642
648395
starting Well D09
22645
671040
starting Well D10
22709
693749
starting Well D11
19746
713495
starting Well F10
24404
737899
starting Well F11
25091
762990
starting Wel

Nutlin

In [4]:
#Builds a dataframe for each well defined in well_list, then outputs the length of that list. The number output is how many cells
#were found in the well

fullest_df = pd.DataFrame()
for well in well_list:
    print(f'starting Well {well}')
    full_df = pd.read_csv(os.path.join(full_dir,f'cell_data_{well}_df.csv'), sep=',') 

   ### This part here is optional, you can add important metadata information at this step. Treatment and Sample ID are shown as examples
    if 'B' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('09' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'B' in well and ('10' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 9
       # full_df['Group'] = 1  
    if 'B' in well and ('11' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1
    
    if 'C' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 2
    if 'C' in well and ('09' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 8
       # full_df['Group'] = 2   
    if 'C' in well and ('10' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 9
       # full_df['Group'] = 1  
    if 'C' in well and ('11' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1 

       
    exec(f'well{well}_df = full_df.copy()')
    fullest_df = pd.concat([fullest_df, full_df], ignore_index = True)
    print(len(full_df))
    print(len(fullest_df))

### You may find that you need to force some of the columns added this way into specific data types. You can do that here.
# Convert sample_ID column to categorical data type
fullest_df['sample_ID'] = fullest_df['sample_ID'].astype('category')
fullest_df['dose'] = fullest_df['dose'].astype(str)
fullest_df['treatment'] = fullest_df['treatment'].astype(str)

starting Well B02
9883
9883
starting Well B09
9186
19069
starting Well B10
6627
25696
starting Well B11
6337
32033
starting Well C02
10803
42836
starting Well C09
10035
52871
starting Well C10
6357
59228
starting Well C11
6592
65820


In [ ]:
#Builds a dataframe for each well defined in well_list, then outputs the length of that list. The number output is how many cells
#were found in the well

fullest_df = pd.DataFrame()
for well in well_list:
    print(f'starting Well {well}')
    full_df = pd.read_csv(os.path.join(full_dir,f'cell_data_{well}_df.csv'), sep=',') 

   ### This part here is optional, you can add important metadata information at this step. Treatment and Sample ID are shown as examples
    if 'B' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    if 'B' in well and ('03' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'B' in well and ('04' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 3
       # full_df['Group'] = 1
    if 'B' in well and ('05' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 4
       # full_df['Group'] = 1
    if 'B' in well and ('06' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 5
        #full_df['Group'] = 1
    if 'B' in well and ('07' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 6
        #full_df['Group'] = 1
    if 'B' in well and ('08' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 7
        #full_df['Group'] = 1
    if 'B' in well and ('09' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 8
       # full_df['Group'] = 1
    if 'B' in well and ('10' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 9
       # full_df['Group'] = 1  
    if 'B' in well and ('11' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1
    if 'B' in well and ('07' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 11
        #full_df['Group'] = 1
    if 'B' in well and ('08' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 11
    
    if 'C' in well and ('02' in well):
        full_df['dose'] = '0'
        full_df['treatment'] = 'Control' 
        full_df['sample_ID'] = 1
       # full_df['Group'] = 2
    if 'C' in well and ('03' in well):
        full_df['dose'] = '10 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 2
    if 'C' in well and ('04' in well):
        full_df['dose'] = '25 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 3
       # full_df['Group'] = 2
    if 'C' in well and ('05' in well):
        full_df['dose'] = '50 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 4
       # full_df['Group'] = 2
    if 'C' in well and ('06' in well):
        full_df['dose'] = '100 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 5
    if 'C' in well and ('07' in well):
        full_df['dose'] = '250 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 6
        #full_df['Group'] = 2
    if 'C' in well and ('08' in well):
        full_df['dose'] = '500 nM'
        full_df['treatment'] = 'Abema' 
        full_df['sample_ID'] = 7
        #full_df['Group'] = 2
    if 'C' in well and ('09' in well):
        full_df['dose'] = '1 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 8
       # full_df['Group'] = 2   
    if 'C' in well and ('10' in well):
        full_df['dose'] = '5 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 9
       # full_df['Group'] = 1  
    if 'C' in well and ('11' in well):
        full_df['dose'] = '10 uM'
        full_df['treatment'] = 'Nutlin' 
        full_df['sample_ID'] = 10
       # full_df['Group'] = 1 

       
    exec(f'well{well}_df = full_df.copy()')
    fullest_df = pd.concat([fullest_df, full_df], ignore_index = True)
    print(len(full_df))
    print(len(fullest_df))

### You may find that you need to force some of the columns added this way into specific data types. You can do that here.
# Convert sample_ID column to categorical data type
fullest_df['sample_ID'] = fullest_df['sample_ID'].astype('category')
fullest_df['dose'] = fullest_df['dose'].astype(str)
fullest_df['treatment'] = fullest_df['treatment'].astype(str)

starting Well B02
9883
9883
starting Well B03
9697
19580
starting Well B04
9705
29285
starting Well B05
9534
38819
starting Well B06
9912
48731
starting Well B07
8906
57637
starting Well B08
9458
67095
starting Well B09
9186
76281
starting Well B10
6627
82908
starting Well B11
6337
89245
starting Well C02
10803
100048
starting Well C03
10596
110644
starting Well C04
10302
120946
starting Well C05
9894
130840
starting Well C06
10027
140867
starting Well C07
9767
150634
starting Well C08
9771
160405
starting Well C09
10035
170440
starting Well C10
6357
176797
starting Well C11
6592
183389


In [6]:
fullest_df.columns

Index(['Unnamed: 0', 'label', 'nuc_area', 'centroid-0', 'centroid-1',
       'orientation', 'major_axis_length', 'minor_axis_length', 'bbox-0',
       'bbox-1', 'bbox-2', 'bbox-3', 'R0_DNA_nuc_mean', 'R0_yH2AX_nuc_mean',
       'R0_53BP1_nuc_mean', 'R1_DNA_nuc_mean', 'R1_p53_nuc_mean',
       'R1_p21_nuc_mean', 'R2_DNA_nuc_mean', 'R2_CHK1_nuc_mean',
       'R2_pCHK1_nuc_mean', 'R3_DNA_nuc_mean', 'R3_pRB_nuc_mean',
       'R3_RB_nuc_mean', 'nuc_mask', 'ring_area', 'R0_DNA_ring_mean',
       'R0_yH2AX_ring_mean', 'R0_53BP1_ring_mean', 'R1_DNA_ring_mean',
       'R1_p53_ring_mean', 'R1_p21_ring_mean', 'R2_DNA_ring_mean',
       'R2_CHK1_ring_mean', 'R2_pCHK1_ring_mean', 'R3_DNA_ring_mean',
       'R3_pRB_ring_mean', 'R3_RB_ring_mean', 'ring_mask',
       'R0_DNA_total_nuc_protein', 'R0_yH2AX_total_nuc_protein',
       'R0_53BP1_total_nuc_protein', 'R1_DNA_total_nuc_protein',
       'R1_p53_total_nuc_protein', 'R1_p21_total_nuc_protein',
       'R2_DNA_total_nuc_protein', 'R2_CHK1_total_nu

In [7]:
### It is good practice to save these dataframes at key steps, such as this one
fullest_df.to_csv(r'D:\Sarah\experiment_4_2\fullest_df.csv')

In [8]:
import pandas as pd

# Identify columns that contain any NaN values
columns_with_nan = fullest_df.columns[fullest_df.isnull().any()].tolist()

# Print the list of column headers with NaN values
print("Columns with NaN values:", columns_with_nan)

Columns with NaN values: []


In [9]:

# Definition to Normalize the dataframe by z-score

def standardizeColumns(df):
    df = df.copy()
    df.iloc[:,:] = df.iloc[:,:].apply(lambda x: (x-x.mean())/ x.std(), axis=0)
    return df

In [12]:
fullest_df["pRB_RB_ratio"] = fullest_df["R3_pRB_nuc_mean"] / fullest_df["R3_RB_nuc_mean"]

In [13]:
fullest_df["pCHK1_CHK1_ratio"] = fullest_df["R2_pCHK1_nuc_mean"] / fullest_df["R2_CHK1_nuc_mean"]

In [30]:
fullest_df["pS6_S6_ratio"] = fullest_df["R1_pS6_nuc_mean"] / fullest_df["R1_S6_nuc_mean"]

KeyError: 'R1_pS6_nuc_mean'

In [14]:
fullest_df["total_DNA"] = fullest_df["R0_DNA_nuc_mean"] * fullest_df["nuc_area"]

In [14]:
fullest_df["total_pS6"] = fullest_df["R1_pS6_nuc_mean"] + fullest_df["R1_pS6_ring_mean"]

In [ ]:
# Drop columns that you don't need - Preparation for conversion to AnnData object
fullest_df = fullest_df.drop(columns=["Unnamed: 0", "bbox-0", "bbox-1", "bbox-2", "bbox-3", "nuc_mask", "ring_mask", "ring_area"])
columns_to_drop = ["Unnamed: 0", "bbox-0", "bbox-1", "bbox-2", "bbox-3", "nuc_mask", "ring_mask", "ring_area"]
# Extract metadata columns and store them in a separate dataframe
metadata = fullest_df[["label", "well", "sample_ID", "treatment", "dose", "centroid-0", "centroid-1"]]
# Remove metadata columns from the main dataframe
fullest_df = fullest_df.drop(columns=["label", "well", "sample_ID", "treatment", "dose", "centroid-0", "centroid-1"])

In [16]:
#Z normalize the data
standard_df = standardizeColumns(fullest_df)

In [17]:
import pandas as pd

# Remove columns that contain any NaN values
print("Shape before removing columns with NaN:", standard_df.shape)
standard_df = standard_df.dropna(axis=1, how='any')

# Print the shape of the DataFrame before and after removing columns with NaN
print("Shape after removing columns with NaN:", standard_df.shape)


Shape before removing columns with NaN: (363672, 56)
Shape after removing columns with NaN: (363672, 44)


In [12]:
#min-max normalize the data - Not used in this example, provided for potential use
'''
scaler = MinMaxScaler()
normalized_data = scaler.fit_transform(fullest_df)
normalized_data = pd.DataFrame(normalized_data, index = fullest_df.index, columns = fullest_df.columns)
'''

'\nscaler = MinMaxScaler()\nnormalized_data = scaler.fit_transform(fullest_df)\nnormalized_data = pd.DataFrame(normalized_data, index = fullest_df.index, columns = fullest_df.columns)\n'

In [18]:
# Convert the pandas dataframe to an anndata object
standard_adata = ad.AnnData(standard_df)
# Add metadata back to the anndata object
standard_adata.obs = metadata.copy()
standard_adata.obs_names = [f'c_{i}' for i in standard_adata.obs_names]

#Save the entire adata file
adata_save_path = r'D:\Sarah\experiment_4_2\standard_adata.h5ad'
standard_adata.write_h5ad(adata_save_path)

c:\Users\Purvis_Jane\.conda\envs\python_3_7\lib\site-packages\ipykernel_launcher.py:2: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  
c:\Users\Purvis_Jane\.conda\envs\python_3_7\lib\site-packages\anndata\_core\anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [19]:
import anndata as ad

# Load the existing AnnData file (do NOT recreate or overwrite)
adata_path = r"D:\Sarah\experiment_4_2\standard_adata.h5ad"
standard_adata = ad.read_h5ad(adata_path)

print(f"Loaded standard AnnData: {standard_adata.shape}")


Loaded standard AnnData: (363672, 44)


In [21]:
###Sketching lets your subsample your data accurately. 
### This example groups for subsampling based on the sample_id metadata

idx, standard_adata_sub = sketch(standard_adata, num_subsamples = 2000, frequency_seed = 42, sample_set_key = 'sample_ID')
#Save the entire adata file
adata_save_path = r'D:\Sarah\experiment_4_2\standard_adata_sub.h5ad'
standard_adata_sub.write_h5ad(adata_save_path)

performing subsampling: 100%|██████████| 10/10 [03:15<00:00, 19.55s/it]


In [ ]:
standard_adata_sub

Subsequent notebooks will load in the saved adata file of your choice, either the full dataset of the subsampled dataset.
Additionally, you can produce any desired normalized or non-normalized adata file here. Z-score normalized is the provided example, but other methods can be used with minor modifications. i.e. min-max normalization using the provided code.